# PKG Staging Transaction Table — EDA

**Payment Knowledge Graph (PKG) · Neo4j staging transaction table**
*PNC Bank · Treasury Management · Data Science*
*Tracks `PKG_TXN_TABLE_EDA_SEED.md` v1 — answers the §6 open questions*

---

## What this notebook does

Phases run in order of consequence, matching §6. **Phase 1 is the blocker**;
nothing downstream of it is trustworthy until it resolves, so it runs first and
on **raw staging rows** — unfiltered, un-exploded, no population gate. The §5
measurement was taken *after* a corporate-ego filter and *after* the ego-leg
explode, which means three candidate causes were never separated:

| Where the duplication could live | Test |
|---|---|
| The customer dimension (`neo4j_customer` fanout on join) | §1.0 |
| The source table itself | §1.1 |
| The explode / reshaping logic | §1.1 by elimination |

**§1.0 has the highest prior and the lowest cost.** If `neo4j_customer` carries
more than one row per `mdm_id`, then every join against it doubles the staging
rows and there is no source defect at all. The seed document does not consider
this branch. It is one `count()` and it runs first.

| Phase | Open question | Output |
|---|---|---|
| 1 | Q1 — `trans_id` duplication | `12b`–`18` in `qa/`, decision in §1.9 |
| 2 | Q3 — history depth | `coverage_monthly.csv` |
| 3 | Q2 — `trans_dt` semantics | `dt_lag_by_rail.csv`, `dt_dow.csv`, `dt_dom.csv` |
| 4 | Q4, Q5, Q6, Q7, Q9 — unprofiled columns | `profile_*.csv` |
| 5 | Q8 — merchant namespace | `merchant_*.csv` |
| 6 | Q10 — non-USD policy | `currency_*.csv` |
| 7 | taxonomy assertions | fails loudly |
| 8 | — | `summary.json` |

## Working rules carried from the seed (§7)

One pass over staging, then never again. `approx_count_distinct` everywhere.
Never `.cache()` anything wide — parquet round-trip. Every `.toPandas()` on an
aggregate only. Cast timestamps to string before Arrow. Two-stage reduce for
anything keyed on counterparty. Work path keyed to the config.

## Before you run

1. Set `TXN_TABLE` in the config cell.
2. Leave `POPULATION = "all"` for Phase 1. Filtering to corporate egos before
   the duplication is understood is what produced the ambiguity in the first
   place.
3. `DIAG_TXN_SAMPLE_PCT` defaults to 5. The sample is taken **on the hashed
   `trans_id`**, so duplicate groups stay whole and group-level statistics are
   unbiased. Set to 100 for the number you brief.

## 0 · Configuration

In [ ]:
# ── Tables ───────────────────────────────────────────────────────────────────
TXN_TABLE = "bdahd01p_dlcdi1_cdi_tm.<STAGING_TXN_TABLE>"   # ← SET THIS
CUST_DIM  = "bdahd01p_dlcdi1_cdi_tm.neo4j_customer"

# ── Scope ────────────────────────────────────────────────────────────────────
# Deep diagnostics (Phases 1, 3-6) run on these months only. One clean month is
# enough to resolve §5; 2025-09 is the month the seed's counts were taken on, so
# results are directly comparable.
PROFILE_MONTHS = ["2025-09"]

# Cheap monthly aggregates (Phase 2). None = discover the full range from the
# table's own min/max trans_dt.
RANGE_MONTHS = None

# Hash-sample of trans_id for the group-level diagnostics in §1.3-§1.6.
# Whole groups are kept or dropped together. 100 = no sampling.
DIAG_TXN_SAMPLE_PCT = 5

# "all" | "corporate". Phase 1 MUST run on "all" — see the header.
POPULATION = "all"

# ── Behaviour ────────────────────────────────────────────────────────────────
REBUILD        = False   # True forces a re-read of staging into the work path
FAIL_ON_TAXONOMY = True  # Phase 7 raises rather than warns
TOP_N          = 25      # rows kept in every "top values" table

# ── Paths — keyed to the config, per §7. A config change must not silently ────
# ── reuse stale output. This was learned the hard way twice. ─────────────────
import os
_tag = f"{'_'.join(PROFILE_MONTHS)}__pop-{POPULATION}__s{DIAG_TXN_SAMPLE_PCT}"
WORK = f"../metrics/txn_eda/{_tag}"
PARQ = f"{WORK}/_staging_slice"          # the one-pass materialisation
QA   = f"{WORK}/qa"
os.makedirs(WORK, exist_ok=True)
os.makedirs(QA, exist_ok=True)
print("work path:", WORK)

### 0.1 Session

Two settings are not optional. Arrow fallback: a `toPandas()` against a
timestamp column killed an early run mid-computation. AQE skew join: hub
counterparties put hundreds of millions of rows behind single keys, and any
`groupBy(cpty_key)` will otherwise hang one task forever.

In [ ]:
from pyspark.sql import SparkSession, functions as F, Window as W
from pyspark.sql.types import TimestampType, DateType, DecimalType
import pandas as pd, numpy as np, json, time, contextlib

SPARK_CONF = {
    "spark.sql.adaptive.enabled": "true",
    "spark.sql.adaptive.skewJoin.enabled": "true",
    "spark.sql.execution.arrow.pyspark.fallback.enabled": "true",
    "spark.sql.shuffle.partitions": "2000",
    "spark.sql.autoBroadcastJoinThreshold": str(64 * 1024 * 1024),
    "spark.sql.sources.partitionOverwriteMode": "static",  # §5.3 — see note
}

_b = SparkSession.builder.appName("pkg_txn_eda")
for k, v in SPARK_CONF.items():
    _b = _b.config(k, v)
spark = _b.enableHiveSupport().getOrCreate()

# If a session already existed, the builder configs above were ignored. Re-set
# the runtime-settable ones explicitly and report what actually took.
for k, v in SPARK_CONF.items():
    try:
        spark.conf.set(k, v)
    except Exception as e:
        print(f"  ! could not set {k}: {e}")
for k in SPARK_CONF:
    print(f"{k:52s} = {spark.conf.get(k, '<unset>')}")

> **§5.3 note.** `partitionOverwriteMode` is pinned to `static` here rather than
> `dynamic`. The second defect in the seed — three months (2025-04/05/06)
> reading ~50% high because a partitioned overwrite *layered* rather than
> replaced — is a dynamic-mode failure. This notebook writes only to
> config-keyed paths and deletes before writing (`_fresh()` below), so it cannot
> reproduce it either way, but the default is set correctly so nothing copied
> out of here inherits the bug.

### 0.2 Helpers

In [ ]:
import shutil

@contextlib.contextmanager
def step(label):
    t0 = time.time()
    print(f"\n── {label} " + "─" * max(0, 66 - len(label)))
    yield
    print(f"   ({time.time() - t0:.1f}s)")


def to_pd(df, n=None):
    """Collect an AGGREGATE. Casts timestamp/date/decimal to string or float
    first — the Arrow fallback in §1 exists because a raw timestamp column
    killed a run mid-computation. Never call this on row-level data."""
    out = df
    for f in df.schema.fields:
        if isinstance(f.dataType, (TimestampType, DateType)):
            out = out.withColumn(f.name, F.col(f.name).cast("string"))
        elif isinstance(f.dataType, DecimalType):
            out = out.withColumn(f.name, F.col(f.name).cast("double"))
    if n:
        out = out.limit(n)
    return out.toPandas()


def save(pdf, name, echo=True):
    path = os.path.join(QA, f"{name}.csv")
    pdf.to_csv(path, index=False)
    print(f"→ {path}  ({len(pdf)} rows)")
    if echo:
        with pd.option_context("display.max_rows", 60, "display.width", 200,
                               "display.max_colwidth", 44):
            display(pdf)
    return pdf


def _fresh(path):
    """Delete before writing. Never rely on overwrite semantics — §5.3."""
    if os.path.isdir(path):
        shutil.rmtree(path)
    elif os.path.exists(path):
        os.remove(path)


# ── column presence: the schema in §2.1 is a claim, not a guarantee ──────────
_RAW = spark.table(TXN_TABLE)
COLS = set(_RAW.columns)

def has(*names):
    return all(n in COLS for n in names)

def cols(*names):
    return [n for n in names if n in COLS]

def present(c):
    """Populated = non-null AND non-blank. NULL and '' are normalised to one
    absent representation, once, here — not by whichever .fillna() runs first."""
    return F.col(c).isNotNull() & (F.trim(F.col(c).cast("string")) != "")


def month_col(c="trans_dt"):
    return F.substring(F.col(c).cast("string"), 1, 7)


RESULTS = {}   # everything headline lands here and is written in Phase 8
def record(k, v):
    RESULTS[k] = v
    return v

### 0.3 Table discovery

Physical layout before logical content. Partition pruning is the difference
between a 3-minute scan and a 3-hour one, and the partition column's own values
are evidence about `trans_dt` (Q2) — if the partition month disagrees with the
`trans_dt` month for a non-trivial share of rows, then one of them is a
*posting* concept and the other is not.

In [ ]:
with step("schema"):
    schema_pd = pd.DataFrame(
        [{"column": f.name, "type": f.dataType.simpleString()} for f in _RAW.schema.fields])
    save(schema_pd, "00_schema", echo=False)
    print(f"{len(COLS)} columns")

    # Columns the seed document says exist
    EXPECTED_COLS = [
        "mdm_id_pays", "mdm_id_pays_orig", "customer_name_pays", "originating_company",
        "mdm_id_receives", "mdm_id_receives_orig", "customer_name_receives",
        "pnc_dep_acct_pays", "pnc_dep_acct_receives", "unq_cpty_id", "unq_cpty_acct_id",
        "cpty_name", "cpty_type", "cpty_fin_entity_name", "trans_id", "trans_amt",
        "trans_currency", "trans_purpose", "trans_dt", "payment_rail", "category",
        "category_prefix", "src_syst", "np_key", "zelle_recipient_token",
        "card_entry_mode", "merchant_id", "merchant_city", "merchant_state",
        "merchant_zip_cd", "merchant_cat_cd", "hdfs_load_ts"]
    missing = [c for c in EXPECTED_COLS if c not in COLS]
    extra   = sorted(COLS - set(EXPECTED_COLS))
    print(f"expected-but-absent : {missing or 'none'}")
    print(f"present-but-undocumented: {extra or 'none'}")
    record("schema_missing", missing)
    record("schema_undocumented", extra)

    # A USD-equivalent column would settle Q10 outright (see Phase 6)
    fx_like = [c for c in COLS if any(t in c.lower()
               for t in ("usd", "base_amt", "amt_usd", "fx", "rate", "equiv"))]
    print(f"FX / USD-equivalent candidates: {fx_like or 'NONE — see Phase 6'}")
    record("fx_columns", fx_like)

In [ ]:
with step("partitions"):
    PART_COLS, PART_SAMPLE = [], []
    try:
        desc = spark.sql(f"DESCRIBE {TXN_TABLE}").collect()
        seen_marker = False
        for r in desc:
            name = (r[0] or "").strip()
            if name.lower().startswith("# partition"):
                seen_marker = True
                continue
            if seen_marker and name and not name.startswith("#"):
                if name not in PART_COLS:
                    PART_COLS.append(name)
    except Exception as e:
        print("DESCRIBE failed:", e)

    try:
        parts = spark.sql(f"SHOW PARTITIONS {TXN_TABLE}").collect()
        PART_SAMPLE = [r[0] for r in parts]
        print(f"{len(PART_SAMPLE)} partitions")
        for p in PART_SAMPLE[:3] + (["  ..."] if len(PART_SAMPLE) > 6 else []) + PART_SAMPLE[-3:]:
            print("  ", p)
    except Exception as e:
        print("SHOW PARTITIONS unavailable:", e)

    print("partition columns:", PART_COLS or "none detected")
    record("partition_columns", PART_COLS)
    record("n_partitions", len(PART_SAMPLE))

In [ ]:
# ── scope(): month pruning. Prefers the partition column when its values look
# ── like months/dates; falls back to a trans_dt prefix filter.

PART_MONTH_COL = None
for c in PART_COLS:
    if any(t in c.lower() for t in ("month", "dt", "date", "time_key", "yyyymm", "part")):
        PART_MONTH_COL = c
        break
print("pruning on:", PART_MONTH_COL or "trans_dt prefix (NO partition pruning — expect a full scan)")


def scope(df, months=None):
    """Single place where month pruning happens. Population gating is a COLUMN
    added in §0.4, not a filter here — a filter would be a join, and a join
    against a non-unique dimension is the leading candidate cause of §5."""
    if months:
        m = list(months)
        if PART_MONTH_COL and PART_MONTH_COL in df.columns:
            pc = F.col(PART_MONTH_COL).cast("string")
            df = df.filter(
                F.substring(pc, 1, 7).isin(m)                    # 2025-09-01 / 2025-09
                | F.substring(F.regexp_replace(pc, "-", ""), 1, 6)
                    .isin([x.replace("-", "") for x in m])       # 202509
            )
        else:
            df = df.filter(month_col("trans_dt").isin(m))
    return df


def topo(df):
    """Direction is a null pattern, not a column — §3.1."""
    p, r = present("mdm_id_pays"), present("mdm_id_receives")
    return df.withColumn(
        "topo",
        F.when(p & r, F.lit("INTERNAL_C2C"))
         .when(r, F.lit("INBOUND"))
         .when(p, F.lit("OUTBOUND"))
         .otherwise(F.lit("ORPHAN")))

### 0.4 The one pass over staging

Project, prune, materialise, and never touch staging again. Every subsequent
cell reads `PARQ`. Measured at 186 s for 17 months in the attrition work, so a
single profile month is cheap — but it is still the only full scan in the
notebook and everything after it is local.

**No population filter is applied here.** Phase 1 needs the raw row population;
the corporate gate is available as a column so later phases can subset without
a second scan.

In [ ]:
KEEP = cols(
    "trans_id", "trans_dt", "trans_amt", "trans_currency", "trans_purpose",
    "mdm_id_pays", "mdm_id_receives", "mdm_id_pays_orig", "mdm_id_receives_orig",
    "customer_name_pays", "customer_name_receives", "originating_company",
    "pnc_dep_acct_pays", "pnc_dep_acct_receives",
    "unq_cpty_id", "unq_cpty_acct_id", "cpty_name", "cpty_type", "cpty_fin_entity_name",
    "payment_rail", "category", "category_prefix", "src_syst", "np_key",
    "zelle_recipient_token", "card_entry_mode",
    "merchant_id", "merchant_city", "merchant_state", "merchant_zip_cd", "merchant_cat_cd",
    "hdfs_load_ts")

if REBUILD or not os.path.exists(PARQ):
    with step(f"one pass over staging → {PARQ}"):
        _fresh(PARQ)
        src = scope(spark.table(TXN_TABLE).select(*KEEP), months=PROFILE_MONTHS)
        src = topo(src).withColumn("month", month_col("trans_dt"))
        # ids are strings end to end (§2.2 dtype trap)
        for c in cols("trans_id", "mdm_id_pays", "mdm_id_receives",
                      "mdm_id_pays_orig", "mdm_id_receives_orig",
                      "unq_cpty_id", "unq_cpty_acct_id", "np_key"):
            src = src.withColumn(c, F.col(c).cast("string"))
        src.write.mode("overwrite").partitionBy("month").parquet(PARQ)
else:
    print(f"reusing {PARQ}  (set REBUILD=True to force)")

TX = spark.read.parquet(PARQ)
N_ROWS = TX.count()
print(f"{N_ROWS:,} raw staging rows over {PROFILE_MONTHS}")
record("n_raw_rows", N_ROWS)

### 0.5 Population gate — a column, not a filter

The corporate-ego gate is attached as a flag so later phases can subset without
a second scan **and** without a join. The distinct-`mdm_id` projection is what
makes it safe: a raw join against `neo4j_customer` is exactly the fanout §1.0
is testing for, and using it here would import the bug into the diagnostic that
is meant to find it.

In [ ]:
with step("0.5 corporate ego flag"):
    dim0 = spark.table(CUST_DIM)
    if "party_type" in dim0.columns:
        corp = (dim0.filter(F.upper(F.trim(F.col("party_type"))) == "O")
                    .select(F.col("mdm_id").cast("string").alias("_m"))
                    .distinct())
        n_corp = corp.count()
        print(f"{n_corp:,} distinct corporate mdm_id (party_type = 'O')")
        # flags, not filters — left joins against a DISTINCT single-column frame
        TX = (TX.join(corp.withColumnRenamed("_m", "mdm_id_pays")
                          .withColumn("_cp", F.lit(1)), on="mdm_id_pays", how="left")
                .join(corp.withColumnRenamed("_m", "mdm_id_receives")
                          .withColumn("_cr", F.lit(1)), on="mdm_id_receives", how="left")
                .withColumn("is_corp_pays", F.coalesce(F.col("_cp"), F.lit(0)))
                .withColumn("is_corp_receives", F.coalesce(F.col("_cr"), F.lit(0)))
                .drop("_cp", "_cr"))
        TX = TX.withColumn("is_corp_ego",
                           F.greatest("is_corp_pays", "is_corp_receives"))
        assert TX.count() == N_ROWS, (
            "the corporate flag join changed the row count — the dimension is not "
            "unique on mdm_id even after .distinct(). Investigate before continuing.")
        print("row count unchanged after flagging — join is safe")
    else:
        print("party_type absent from the dimension; skipping the corporate flag")
        TX = TX.withColumn("is_corp_ego", F.lit(None).cast("int"))


def corporate(df):
    """Subset to corporate egos. Phase 1 must NOT use this."""
    return df.filter(F.col("is_corp_ego") == 1)

---

# Phase 1 · The blocking defect — why is `trans_id` duplicated?

**Open question 1.** Everything that counts or sums depends on this.

The seed's arithmetic (§5) is sound *given* that the input to the explode was
one row per transaction. It reasons: internal transactions are 6% of the data
and produce two legs each, so the expected legs-per-transaction is ~1.06; the
measured 2.0 must therefore come from the source. That inference is only valid
if nothing between the source table and the leg table multiplied rows. Two
things sat in that gap — the corporate-ego filter (a **join**, §1.0) and the
explode itself. Both are tested before the source is blamed.

## 1.0 · Is the customer dimension unique?

Highest prior, lowest cost. A join to `neo4j_customer` that fans out doubles
every staging row and produces exactly the symptom in §5 — with no defect in
the source table at all. The seed already records that `mdm_id → cust_pwr_id`
is *near* 1:1 (152 egos with two, 1 with three) but that is the wrong direction:
the fanout that matters on a staging join is **rows per `mdm_id`** in the
dimension.

In [ ]:
with step("1.0 dimension uniqueness"):
    dim = spark.table(CUST_DIM)
    for c in ["mdm_id", "cust_pwr_id"]:
        if c in dim.columns:
            dim = dim.withColumn(c, F.col(c).cast("string"))

    n_dim   = dim.count()
    n_mdm   = dim.select("mdm_id").distinct().count()
    fanout  = n_dim / max(n_mdm, 1)
    print(f"neo4j_customer rows      : {n_dim:,}")
    print(f"distinct mdm_id          : {n_mdm:,}")
    print(f"rows per mdm_id (mean)   : {fanout:.4f}")

    rpm = (dim.groupBy("mdm_id").count()
              .groupBy("count").agg(F.count(F.lit(1)).alias("n_mdm_id"))
              .orderBy("count"))
    save(to_pd(rpm), "10_dim_rows_per_mdm_id")

    if "cust_pwr_id" in dim.columns:
        pw = (dim.filter(present("cust_pwr_id"))
                 .groupBy("cust_pwr_id").agg(F.countDistinct("mdm_id").alias("n_mdm"))
                 .groupBy("n_mdm").agg(F.count(F.lit(1)).alias("n_cust_pwr_id"))
                 .orderBy("n_mdm"))
        save(to_pd(pw), "10_dim_mdm_per_cust_pwr_id")

    record("dim_rows_per_mdm_id", round(fanout, 4))
    if fanout > 1.02:
        print(f"\n*** DIMENSION FANS OUT AT {fanout:.2f}x ***")
        print("    A join against this table multiplies staging rows by ~this factor.")
        print("    This is a sufficient explanation for §5 on its own. Confirm in §1.1:")
        print("    if raw staging is clean, the defect is the join, not the source.")
    else:
        print("\n    Dimension is effectively unique. The join is not the cause.")

## 1.1 · The decisive test — raw staging, no filter, no explode

If `rows / distinct(trans_id)` is ≈2.0 on **raw** staging, the duplication is in
the source and the seed's §5 conclusion stands. If it is ≈1.0, the source is
clean and the defect was introduced downstream — by the join in §1.0 or by the
explode. There is no third reading of this number.

`approx_count_distinct` runs first (an exact `countDistinct` on `trans_id`
against unfiltered staging killed an early run). The exact count is then taken
on the profile slice only, where it is affordable, because the ratio is the
number that decides the programme and a 2% approximation error is not
acceptable at the 1.0-vs-2.0 boundary.

In [ ]:
with step("1.1 rows vs distinct trans_id — raw"):
    approx = TX.select(F.approx_count_distinct("trans_id").alias("d")).first()["d"]
    exact  = TX.select("trans_id").distinct().count()
    ratio  = N_ROWS / max(exact, 1)
    print(f"raw rows                 : {N_ROWS:,}")
    print(f"distinct trans_id (approx): {approx:,}")
    print(f"distinct trans_id (exact) : {exact:,}")
    print(f"rows per trans_id         : {ratio:.4f}")
    record("raw_rows_per_trans_id", round(ratio, 4))

    print()
    if ratio > 1.5:
        print("  → SOURCE DUPLICATION CONFIRMED. The seed's §5 inference holds.")
        print("    Continue to §1.2-§1.8 to establish the mechanism.")
    elif ratio < 1.05:
        print("  → SOURCE IS CLEAN. trans_id is ~unique on the raw table.")
        print("    The duplication was introduced downstream. Re-read §1.0; if the")
        print("    dimension is also clean, the explode is the remaining suspect and")
        print("    §1.2-§1.8 will be uninformative — go straight to the leg builder.")
    else:
        print("  → PARTIAL. Some duplication, not a clean 2x. §1.4's signature table")
        print("    will say whether one mechanism or several are mixed.")

## 1.2 · Rows per `trans_id` — the raw distribution

The seed's table (1 / **2** / 3 / 4) was measured on legs. This is the same
distribution on source rows, which is what the fix has to be written against.
A clean 1-and-2 with a small 4 tail means one duplication mechanism plus the
internal-topology interaction. A long tail means something else.

In [ ]:
with step("1.2 rows per trans_id distribution"):
    grp = TX.groupBy("trans_id").agg(
        F.count(F.lit(1)).alias("n_rows"),
        F.sum(F.col("trans_amt").cast("double")).alias("grp_amt"))
    dist = (grp.groupBy("n_rows")
               .agg(F.count(F.lit(1)).alias("n_trans_id"),
                    F.sum("grp_amt").alias("dollars"))
               .orderBy("n_rows"))
    d = to_pd(dist)
    d["rows"]        = d.n_rows * d.n_trans_id
    d["share_rows"]  = d.rows / d.rows.sum()
    d["share_dollars"] = d.dollars / d.dollars.sum()
    save(d, "12_rows_per_trans_id")
    record("rows_per_trans_id_hist",
           {int(r.n_rows): int(r.n_trans_id) for r in d.itertuples()})

## 1.2b · Who duplicates? — duplication rate by dimension

Composition of the duplicated population, over the **full slice** (no sampling,
no join). For each value of each dimension: rows, distinct transactions, and the
ratio between them.

**Read this table backwards.** The obvious reading is "which rail duplicates
most". The more useful one is the opposite:

> A dimension whose within-value ratio is **≈1.0 everywhere** while the overall
> ratio is ≈2.0 is *the axis the duplicate rows straddle*.

If every `src_syst` value shows 1.0 rows per transaction but the table overall
shows 2.0, then each transaction appears exactly once *per source system* — the
two rows carry different `src_syst` values and the duplication is a two-system
capture. If instead every rail shows ≈2.0, the duplication happens **inside** a
rail and `payment_rail` is not the splitting axis.

That single contrast identifies the mechanism before any group-level work runs,
and it is the cheapest test in Phase 1 after §1.1.

In [ ]:
DUP_DIMS = cols("topo", "payment_rail", "category_prefix", "src_syst",
                "cpty_type", "trans_currency", "is_corp_ego")

with step("1.2b duplication rate by dimension"):
    frames = []
    for dim in DUP_DIMS:
        t = (TX.groupBy(F.coalesce(F.col(dim).cast("string"), F.lit("<NULL>"))
                         .alias("value"))
               .agg(F.count(F.lit(1)).alias("rows"),
                    F.approx_count_distinct("trans_id").alias("approx_txns"),
                    F.sum(F.col("trans_amt").cast("double")).alias("dollars")))
        p = to_pd(t)
        p.insert(0, "dimension", dim)
        p["rows_per_txn"]  = p.rows / p.approx_txns.clip(lower=1)
        p["share_of_rows"] = p.rows / N_ROWS
        frames.append(p.sort_values("rows", ascending=False))
    dd = pd.concat(frames, ignore_index=True)
    save(dd, "12b_duplication_by_dimension")

    overall = record("raw_rows_per_trans_id", RESULTS.get("raw_rows_per_trans_id"))
    print(f"\noverall rows per trans_id: {overall}")
    print("\nDimensions where EVERY value sits near 1.0 while the overall ratio is")
    print("well above it — these are the candidate splitting axes:")
    flat = (dd[dd.share_of_rows > 0.001]
              .groupby("dimension").rows_per_txn.max().reset_index()
              .rename(columns={"rows_per_txn": "max_within_value_ratio"}))
    flat["straddles"] = flat.max_within_value_ratio < (0.6 * (overall or 2.0) + 0.4)
    display(flat.sort_values("max_within_value_ratio"))
    record("splitting_axis_candidates", flat[flat.straddles].dimension.tolist())

## 1.3 · Agreement profile — which columns differ inside a duplicate group

The seed proposes reading one duplicated `trans_id` and eyeballing five columns
(§5.2). One example cannot distinguish "this is the mechanism" from "this is
one of several mechanisms", and the fix differs by branch. This measures the
same thing across every duplicate group at once.

**Method.** Per group and per column: `min`, `max`, and a null count. A column
*differs* when `min != max`, or when it is null on some rows of the group and
populated on others. This is exact for the question "does anything disagree"
and costs one shuffle, where a per-column `countDistinct` would expand the
grouping set twenty-odd times.

Sampling is on `hash(trans_id)`, so groups are kept whole and the per-group
statistics are unbiased.

In [ ]:
# Low-cardinality columns whose VALUE SET inside a duplicate group is worth
# keeping — this is what makes the pair tables in §1.4b possible.
PATTERN_COLS = cols("topo", "src_syst", "category_prefix", "payment_rail",
                    "cpty_type", "trans_currency")

DIFF_COLS = cols(
    "src_syst", "category", "category_prefix", "payment_rail",
    "trans_amt", "trans_dt", "trans_currency", "trans_purpose",
    "mdm_id_pays", "mdm_id_receives", "mdm_id_pays_orig", "mdm_id_receives_orig",
    "pnc_dep_acct_pays", "pnc_dep_acct_receives",
    "unq_cpty_acct_id", "unq_cpty_id", "cpty_name", "cpty_type",
    "cpty_fin_entity_name", "originating_company", "np_key",
    "merchant_id", "hdfs_load_ts", "topo")

SAMP = TX
if DIAG_TXN_SAMPLE_PCT < 100:
    SAMP = TX.filter((F.abs(F.hash("trans_id")) % 100) < DIAG_TXN_SAMPLE_PCT)
    print(f"group diagnostics on a {DIAG_TXN_SAMPLE_PCT}% hash-sample of trans_id")

GRP_PARQ = f"{WORK}/_dup_groups"

if REBUILD or not os.path.exists(GRP_PARQ):
    with step("1.3 agreement profile"):
        aggs = [F.count(F.lit(1)).alias("n_rows"),
                F.sum(F.col("trans_amt").cast("double")).alias("grp_amt")]
        for c in PATTERN_COLS:
            # NULL must be VISIBLE in the pattern. src_syst is ~97% null, so a
            # (null, PME) pair is the single most likely two-system capture --
            # and a bare collect_set would silently drop the null side of it,
            # rendering exactly the mechanism we are hunting for as "PME" and
            # indistinguishable from two PME rows.
            v = F.coalesce(F.col(c).cast("string"), F.lit("<NULL>"))
            aggs.append(F.sort_array(F.collect_set(v)).alias(f"_set_{c}"))
        for c in DIFF_COLS:
            s = F.col(c).cast("string")
            aggs += [F.min(s).alias(f"_mn_{c}"),
                     F.max(s).alias(f"_mx_{c}"),
                     F.sum(F.col(c).isNull().cast("int")).alias(f"_nu_{c}")]
        g = SAMP.groupBy("trans_id").agg(*aggs)

        for c in DIFF_COLS:
            differs = (F.coalesce(F.col(f"_mn_{c}") != F.col(f"_mx_{c}"), F.lit(False))
                       | ((F.col(f"_nu_{c}") > 0) & (F.col(f"_nu_{c}") < F.col("n_rows"))))
            g = g.withColumn(f"d_{c}", differs.cast("int"))

        g = g.select("trans_id", "n_rows", "grp_amt",
                     *[F.concat_ws("|", f"_set_{c}").alias(f"p_{c}")
                       for c in PATTERN_COLS],
                     *[f"d_{c}" for c in DIFF_COLS])
        g = g.withColumn("topo_pattern", F.col("p_topo"))
        _fresh(GRP_PARQ)
        # narrow frame — parquet round-trip rather than .cache(), per §7
        g.write.mode("overwrite").parquet(GRP_PARQ)

G = spark.read.parquet(GRP_PARQ)
DUP = G.filter(F.col("n_rows") > 1)
n_dup = DUP.count()
print(f"{n_dup:,} duplicate groups in the diagnostic sample")

In [ ]:
with step("1.3 which columns disagree"):
    row = DUP.agg(*[F.avg(f"d_{c}").alias(c) for c in DIFF_COLS]).first().asDict()
    ag = (pd.DataFrame({"column": list(row.keys()),
                        "share_of_dup_groups_differing": list(row.values())})
            .sort_values("share_of_dup_groups_differing", ascending=False)
            .reset_index(drop=True))
    save(ag, "13_agreement_profile")
    record("agreement_profile",
           {r.column: round(float(r.share_of_dup_groups_differing), 4)
            for r in ag.itertuples()})

    stable = ag[ag.share_of_dup_groups_differing < 0.001].column.tolist()
    print(f"\ncolumns identical in >99.9% of duplicate groups:\n  {stable}")
    print("\nRead the top of the table, not the bottom. The column that differs in")
    print("nearly every duplicate group IS the mechanism. If nothing differs, the")
    print("rows are exact duplicates and dropDuplicates() is the whole fix.")

## 1.4 · Signature classification — one mechanism or several?

Each duplicate group gets a signature: the set of columns that disagree within
it. Grouping on the signature converts the seed's three-branch decision table
into a measured distribution, with dollars attached, so the answer is *which
fix applies to what share of the table* rather than *which story the one
sampled row tells*.

In [ ]:
with step("1.4 signature table"):
    sig = F.concat_ws("+", *[F.when(F.col(f"d_{c}") == 1, F.lit(c)) for c in DIFF_COLS])
    S = (DUP.withColumn("signature", F.when(sig == "", F.lit("<IDENTICAL ROWS>")).otherwise(sig))
            .groupBy("n_rows", "signature")
            .agg(F.count(F.lit(1)).alias("n_groups"),
                 F.sum("grp_amt").alias("dollars"))
            .orderBy(F.desc("n_groups")))
    s = to_pd(S.limit(60))
    s["share_of_dup_groups"] = s.n_groups / n_dup
    s["cum_share"] = s.share_of_dup_groups.cumsum()
    save(s, "14_dup_signatures")

    top = s.iloc[0]
    print(f"\ndominant signature covers {top.share_of_dup_groups:.1%} of duplicate groups:")
    print(f"  n_rows={top.n_rows}  [{top.signature}]")
    record("dominant_signature", str(top.signature))
    record("dominant_signature_share", round(float(top.share_of_dup_groups), 4))
    print(f"\ntop 3 signatures cover {s.cum_share.iloc[min(2, len(s)-1)]:.1%}")
    print("Below ~90% cumulative in the top 3, several mechanisms are mixed and a")
    print("single dedup rule will be wrong for part of the table.")

## 1.4b · Which *values* pair up

§1.4 says a column disagrees. This says what it disagrees between. For each
low-cardinality column, the sorted set of values present inside the group,
counted and dollar-weighted.

`<NULL>` is a first-class value here. With `src_syst` ~97% null (§3.5), the
expected shape of a two-system capture is the pair `<NULL>|PME` — one row from
the ACH/CHECK path that carries no source system, one from the wire path that
does. A `collect_set` that dropped nulls would render that group as `PME` and
hide the mechanism completely, which is why the coalesce in §1.3 is not
cosmetic.

Rows where `is_split` is False are groups whose two rows agree on that column —
so the split, if any, is somewhere else.

In [ ]:
with step("1.4b value pairs inside duplicate groups"):
    pair_summary = []
    for c in PATTERN_COLS:
        t = (DUP.groupBy(f"p_{c}")
                .agg(F.count(F.lit(1)).alias("n_groups"),
                     F.sum("grp_amt").alias("dollars"))
                .orderBy(F.desc("n_groups")))
        p = to_pd(t.limit(30))
        p["share_of_dup_groups"] = p.n_groups / n_dup
        p["is_split"] = p[f"p_{c}"].str.contains("|", regex=False)
        p.to_csv(os.path.join(QA, f"14b_value_pairs_{c}.csv"), index=False)
        split_share = float(p.loc[p.is_split, "share_of_dup_groups"].sum())
        split_dollars = float(p.loc[p.is_split, "dollars"].sum())
        pair_summary.append({"column": c, "share_of_dup_groups_split": split_share,
                             "dollars_in_split_groups": split_dollars})
        print(f"\n── {c}: {split_share:.1%} of duplicate groups span >1 value "
              + "─" * max(0, 20 - len(c)))
        display(p.head(10))

    ps = pd.DataFrame(pair_summary).sort_values("share_of_dup_groups_split",
                                                ascending=False)
    save(ps, "14b_pair_summary")
    record("pair_split_share",
           {r.column: round(float(r.share_of_dup_groups_split), 4)
            for r in ps.itertuples()})

## 1.4c · Mechanism × dimension

The cross-tab the seed's decision table implies but does not provide: for each
duplication signature, which rails, prefixes and topologies carry it. This is
what says "the two-system split is 98% WIRE and the identical-row duplication is
all ACH" — i.e. whether one fix can be applied globally or the table needs a
per-slice rule.

No join back to rows is needed: §1.3 already carried the dimension patterns onto
the group table.

In [ ]:
with step("1.4c mechanism by dimension"):
    sig = F.concat_ws("+", *[F.when(F.col(f"d_{c}") == 1, F.lit(c)) for c in DIFF_COLS])
    LAB = DUP.withColumn("signature",
                         F.when(sig == "", F.lit("<IDENTICAL ROWS>")).otherwise(sig))

    top_sigs = [r["signature"] for r in
                LAB.groupBy("signature").count().orderBy(F.desc("count"))
                   .limit(6).collect()]
    LAB = LAB.withColumn("signature",
                         F.when(F.col("signature").isin(top_sigs), F.col("signature"))
                          .otherwise(F.lit("<other>")))

    for c in [x for x in PATTERN_COLS if x != "trans_currency"]:
        t = to_pd(LAB.groupBy("signature", f"p_{c}")
                     .agg(F.count(F.lit(1)).alias("n_groups"),
                          F.sum("grp_amt").alias("dollars"))
                     .orderBy(F.desc("n_groups")).limit(120))
        piv = t.pivot_table(index="signature", columns=f"p_{c}",
                            values="n_groups", aggfunc="sum", fill_value=0)
        share = piv.div(piv.sum(axis=1), axis=0).round(3)
        print(f"\n── signature × {c}  (row-normalised share of groups) " + "─" * 10)
        with pd.option_context("display.width", 220, "display.max_columns", 30):
            display(share)
        share.reset_index().to_csv(
            os.path.join(QA, f"14c_signature_by_{c}.csv"), index=False)

## 1.5 · Topology inside the duplicate group — the consequential branch

The seed's third branch is framed on `pnc_dep_acct_*`: if the two rows are the
two ledger sides of one movement, the row *is* the leg and the explode
double-counts. There is a sharper version of that test.

If a duplicate pair is `(pays=X, receives=null)` on one row and
`(pays=null, receives=Y)` on the other, then what the null-pattern model reads
as **one OUTBOUND and one INBOUND is actually a single INTERNAL_C2C
transaction split across two source rows**. That would mean §3.1's topology mix
is wrong, the internal share is understated, and the PKG's 6%-of-rows /
19.4%-of-dollars coverage claim is understated with it.

`OUTBOUND|INBOUND` appearing as a common pattern here is the finding. A single
repeated class (`OUTBOUND|OUTBOUND` collapsing to `OUTBOUND`) is not — that is
plain duplication.

In [ ]:
with step("1.5 topology pattern within duplicate groups"):
    tp = (DUP.groupBy("n_rows", "topo_pattern")
             .agg(F.count(F.lit(1)).alias("n_groups"),
                  F.sum("grp_amt").alias("dollars"))
             .orderBy(F.desc("n_groups")))
    t = to_pd(tp.limit(40))
    t["share"] = t.n_groups / n_dup
    save(t, "15_dup_topology_pattern")

    mixed = t[t.topo_pattern.str.contains(r"\|")]
    mixed_share = float(mixed.share.sum()) if len(mixed) else 0.0
    record("dup_mixed_topology_share", round(mixed_share, 4))
    print(f"\nduplicate groups spanning MORE THAN ONE topology class: {mixed_share:.2%}")
    if mixed_share > 0.05:
        print("\n*** The two rows are two SIDES, not two copies. ***")
        print("    The ego-leg model is double-counting non-internal topologies and")
        print("    §3.1's topology mix is measured on a split view of the same")
        print("    transactions. Do not patch — rebuild the leg model, and rename")
        print("    'leg' first (§0 of the seed: it collides with ledger usage).")
    else:
        print("\n    Duplicate rows stay within one topology class. The explode is")
        print("    not the mechanism; the rows are copies of one side.")

## 1.6 · What key *is* unique?

The practical output. If some composite key achieves rows == distinct, that key
is the transaction identity and the fix is a `dropDuplicates` on it (or a
documented precedence rule, if the extra columns carry meaning). `np_key` is
included as a candidate on its own — Q6 asks whether it is an identifier, and
if it is, it may already be the key the table lacks.

In [ ]:
CANDIDATE_KEYS = [k for k in [
    ["trans_id"],
    ["np_key"],
    ["trans_id", "src_syst"],
    ["trans_id", "category"],
    ["trans_id", "category_prefix"],
    ["trans_id", "topo"],
    ["trans_id", "trans_dt"],
    ["trans_id", "trans_amt"],
    ["trans_id", "np_key"],
    ["trans_id", "pnc_dep_acct_pays", "pnc_dep_acct_receives"],
    ["trans_id", "mdm_id_pays", "mdm_id_receives"],
    ["trans_id", "unq_cpty_acct_id"],
    ["trans_id", "hdfs_load_ts"],
] if all(c in TX.columns or c == "topo" for c in k)]

with step("1.6 candidate key search"):
    out = []
    for k in CANDIDATE_KEYS:
        kk = F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000"))
                                     for c in k])
        d = TX.select(kk.alias("k")).select(F.approx_count_distinct("k").alias("d")).first()["d"]
        out.append({"key": " + ".join(k), "approx_distinct": d,
                    "rows_per_key": N_ROWS / max(d, 1)})
    ks = pd.DataFrame(out).sort_values("rows_per_key").reset_index(drop=True)

    # exact count only for the candidates that look unique — approx carries ~2%
    for i, r in ks.iterrows():
        if r.rows_per_key < 1.05:
            k = [c.strip() for c in r.key.split("+")]
            kk = F.concat_ws("\u0001", *[F.coalesce(F.col(c).cast("string"), F.lit("\u0000"))
                                         for c in k])
            e = TX.select(kk.alias("k")).distinct().count()
            ks.loc[i, "exact_distinct"] = e
            ks.loc[i, "rows_per_key"] = N_ROWS / max(e, 1)
    save(ks, "16_candidate_keys")

    winner = ks.iloc[0]
    record("best_key", str(winner.key))
    record("best_key_rows_per_key", round(float(winner.rows_per_key), 4))
    print(f"\ntightest key: {winner.key}  →  {winner.rows_per_key:.4f} rows/key")
    if winner.rows_per_key >= 1.05:
        print("    NO candidate key is unique. The duplication is not resolvable by")
        print("    keying — which points at the ledger-sides reading in §1.5.")

## 1.7 · The examples — stratified, not sampled at random

The seed's §5.2 diagnostic, run once per dominant signature rather than once
overall. Reading one example per mechanism is informative; reading one example
overall is a coin flip.

In [ ]:
SHOW_COLS = [c for c in TX.columns if c not in ("month",)]

def show_diff(tid, note=""):
    """Every raw row for one trans_id, side by side, with the columns that
    actually differ marked. Reading two transposed frames and eyeballing them
    against each other is how a differing column gets missed."""
    ex = to_pd(TX.filter(F.col("trans_id") == tid).select(*SHOW_COLS))
    if len(ex) < 2:
        print(f"  {tid}: only {len(ex)} row(s)")
        return
    t = ex.T
    t.columns = [f"row_{i + 1}" for i in range(t.shape[1])]
    # NULL must be a comparable VALUE here, exactly as in the collect_set in
    # §1.3. pandas' nunique drops NaN by default, so a column that is null on
    # one row and populated on the other would otherwise be marked identical --
    # which is precisely the shape a src_syst split takes.
    norm = t.where(t.notna(), "<NULL>").astype(str)
    t.insert(0, "", np.where(norm.nunique(axis=1) > 1, ">>> DIFFERS", ""))
    print(f"\ntrans_id = {tid}   {note}")
    with pd.option_context("display.max_rows", None, "display.width", 240,
                           "display.max_colwidth", 46):
        display(t)


with step("1.7 side-by-side examples, one per mechanism"):
    sig = F.concat_ws("+", *[F.when(F.col(f"d_{c}") == 1, F.lit(c)) for c in DIFF_COLS])
    labelled = DUP.withColumn(
        "signature", F.when(sig == "", F.lit("<IDENTICAL ROWS>")).otherwise(sig))

    top_sigs = [r["signature"] for r in
                labelled.groupBy("signature").count().orderBy(F.desc("count"))
                        .limit(4).collect()]

    for sg in top_sigs:
        ids = [r["trans_id"] for r in
               labelled.filter(F.col("signature") == sg)
                       .select("trans_id").limit(2).collect()]
        print("\n" + "=" * 78)
        print(f"SIGNATURE: {sg}")
        print("=" * 78)
        for tid in ids:
            show_diff(tid)

## 1.7b · The two-source hypothesis, examined directly

If one payment is captured by two systems, the pair is visible in
`p_src_syst` and the interesting question is what *else* moves with it — whether
the two rows agree on amount, counterparty and deposit account (one payment,
two captures) or disagree on them (two different things that happen to share an
id).

Examples are drawn per **value pair**, so each distinct capture combination gets
its own look rather than the sample landing twice on the same one. The same
treatment is applied to `category_prefix`, since `category` carries the
origination channel that `src_syst` does not (§3.3) and either could be the
axis.

In [ ]:
with step("1.7b examples by value pair"):
    for c in [x for x in ("src_syst", "category_prefix", "payment_rail", "cpty_type")
              if x in PATTERN_COLS]:
        split = DUP.filter(F.col(f"p_{c}").contains("|"))
        pairs = [r[f"p_{c}"] for r in
                 split.groupBy(f"p_{c}").count().orderBy(F.desc("count"))
                      .limit(3).collect()]
        if not pairs:
            print(f"\n{c}: no duplicate group spans more than one value — "
                  f"not the splitting axis")
            continue
        print("\n" + "#" * 78)
        print(f"# {c} — top {len(pairs)} value pairs")
        print("#" * 78)
        for pr in pairs:
            ids = [r["trans_id"] for r in
                   split.filter(F.col(f"p_{c}") == pr)
                        .select("trans_id").limit(1).collect()]
            for tid in ids:
                show_diff(tid, note=f"[{c} = {pr}]")

## 1.8 · Dollar impact

§5.1 says all dollar totals are "roughly 2×". This measures it, per rail, so the
correction factor is known per slice rather than assumed uniform. If the factor
varies by rail, no single global adjustment will fix historical numbers.

In [ ]:
with step("1.8 dollar impact by rail"):
    raw = (TX.groupBy("payment_rail")
             .agg(F.count(F.lit(1)).alias("rows"),
                  F.sum(F.col("trans_amt").cast("double")).alias("raw_dollars")))
    # dedup-by-trans_id estimate: one amount per trans_id per rail
    ded = (TX.groupBy("payment_rail", "trans_id")
             .agg(F.max(F.col("trans_amt").cast("double")).alias("a"))
             .groupBy("payment_rail")
             .agg(F.count(F.lit(1)).alias("txns"),
                  F.sum("a").alias("dedup_dollars")))
    imp = to_pd(raw.join(ded, "payment_rail", "outer").orderBy(F.desc("raw_dollars")))
    imp["dollar_inflation"] = imp.raw_dollars / imp.dedup_dollars
    imp["row_inflation"]    = imp.rows / imp.txns
    save(imp, "18_dollar_impact_by_rail")
    record("dollar_inflation_overall",
           round(float(imp.raw_dollars.sum() / imp.dedup_dollars.sum()), 4))
    print("\n  'dedup_dollars' assumes duplicate rows carry the SAME amount — check")
    print("  d_trans_amt in §1.3 before using it. If amounts differ inside a group,")
    print("  max() is the wrong reducer and this table is an upper bound only.")

## 1.9 · Decision

Read the four numbers, in this order. The first one that fires decides.

| Test | Reading | Fix |
|---|---|---|
| §1.0 `dim_rows_per_mdm_id` > 1.02 **and** §1.1 ratio ≈ 1.0 | The join fanned out. No source defect | `dropDuplicates` on the dimension before joining, or aggregate it to one row per `mdm_id`. Re-run the attrition panel |
| §1.3 nothing differs, §1.5 single-class | Exact duplicates | `dropDuplicates(["trans_id"])` on read |
| §1.2b some dimension is ≈1.0 within every value; §1.4b shows a dominant `<NULL>\|PME`-style pair | One payment captured by two systems | Documented precedence rule — **do not dedup blindly**. §1.4b gives the pair, §1.4c the slice it applies to, §1.7b the raw rows |
| §1.5 `dup_mixed_topology_share` > 5%, or §1.6 finds no unique key | The two rows are the two ledger sides | The row **is** the leg. Rebuild the model, rename `leg` → `ego_row` first |

Whatever the answer, §1.8 gives the correction factor per rail, so the numbers
already briefed can be restated rather than withdrawn.

---

# Phase 2 · How far back does the table go?

**Open question 3.** Profiled from 2025-01 only; earlier coverage unknown and it
gates every longitudinal claim. Three things matter and only one of them is the
min date: a month that *exists* but is thin, or whose distinct-customer count
steps, is not usable history even though it returns rows.

This phase runs over `RANGE_MONTHS` — the full table, not the profile slice —
but only cheap aggregates, no collection of row-level data.

In [ ]:
with step("2.1 date range"):
    rng = spark.table(TXN_TABLE).select(
        F.min("trans_dt").cast("string").alias("min_dt"),
        F.max("trans_dt").cast("string").alias("max_dt")).first()
    print(f"trans_dt spans {rng['min_dt']} → {rng['max_dt']}")
    record("trans_dt_min", rng["min_dt"])
    record("trans_dt_max", rng["max_dt"])

In [ ]:
with step("2.2 monthly coverage"):
    base = spark.table(TXN_TABLE)
    if RANGE_MONTHS:
        base = scope(base, months=RANGE_MONTHS)
    cov = (base.withColumn("month", month_col("trans_dt"))
               .groupBy("month")
               .agg(F.count(F.lit(1)).alias("rows"),
                    F.approx_count_distinct("trans_id").alias("approx_txns"),
                    F.sum(F.col("trans_amt").cast("double")).alias("dollars"),
                    F.approx_count_distinct("mdm_id_pays").alias("payers"),
                    F.approx_count_distinct("mdm_id_receives").alias("receivers"),
                    F.approx_count_distinct("unq_cpty_acct_id").alias("cpty_keys"),
                    F.min("trans_dt").cast("string").alias("first_dt"),
                    F.max("trans_dt").cast("string").alias("last_dt"),
                    F.approx_count_distinct("trans_dt").alias("n_days"))
               .orderBy("month"))
    c = to_pd(cov)
    c["rows_per_txn"] = c.rows / c.approx_txns
    c["rows_mom"] = c.rows.pct_change()
    save(c, "20_coverage_monthly")

    print("\nMonths where rows move >30% against the prior month — a data-availability")
    print("step, not a business trend. §5.3's layered-partition defect looks like this:")
    display(c[c.rows_mom.abs() > 0.30][["month", "rows", "rows_mom", "n_days"]])

    print("\nMonths with fewer than 28 distinct trans_dt values (partial months):")
    display(c[c.n_days < 28][["month", "n_days", "first_dt", "last_dt", "rows"]])
    record("months_covered", c.month.tolist())

The `rows_per_txn` column is the §5 ratio computed per month. If it is flat
across the whole history, the duplication is structural and has always been
there. If it steps at a date, it was introduced by an upstream change — and
that date is the single most useful thing to hand the source-system team.

---

# Phase 3 · What does `trans_dt` mean?

**Open question 2.** The seed concludes this may need an upstream answer because
there is no second date column to compare against. There is one:
**`hdfs_load_ts`**. It is a pipeline timestamp, not a business date, but the
*distribution of the gap* between them is diagnostic, and two further tests need
no second column at all.

| Test | Initiation / transaction date | Settlement / posting date |
|---|---|---|
| **Load lag by rail** | Rail-dependent: wire ≈ same day, ACH 1–3 d, check longest | Flat across rails — everything posts, then loads |
| **Negative lag** | Impossible — cannot load before it happens | Possible: a forward-dated effective date loads early |
| **Weekend mass** | Present on 24/7 rails (RTP, card), absent on batch rails | Near-zero on **every** rail — banking days only |

The weekend test is the strongest of the three. If RTP and card carry normal
Saturday volume while ACH and CHECK carry none, `trans_dt` is a posting date on
the batch rails and an event date on the real-time ones — which is worse than
either single answer, because it means the column's semantics vary by rail and
any daily-grain sequence claim is measuring the settlement pipeline for 95% of
the data.

In [ ]:
with step("3.1 load lag by rail"):
    if not has("hdfs_load_ts"):
        print("no hdfs_load_ts — skipping")
    else:
        lag = (TX.withColumn("txn_d", F.to_date(F.col("trans_dt").cast("string")))
                 .withColumn("load_d", F.to_date(F.col("hdfs_load_ts")))
                 .withColumn("lag_d", F.datediff("load_d", "txn_d"))
                 .filter(F.col("lag_d").isNotNull()))
        L = (lag.groupBy("payment_rail")
                .agg(F.count(F.lit(1)).alias("rows"),
                     F.avg((F.col("lag_d") < 0).cast("int")).alias("share_negative_lag"),
                     F.percentile_approx("lag_d", [0.01, 0.25, 0.5, 0.75, 0.99]).alias("p"),
                     F.min("lag_d").alias("min_lag"), F.max("lag_d").alias("max_lag"))
                .orderBy(F.desc("rows")))
        l = to_pd(L)
        for i, q in enumerate(["p01", "p25", "p50", "p75", "p99"]):
            l[q] = l.p.apply(lambda v, i=i: v[i] if v is not None and len(v) > i else None)
        l = l.drop(columns=["p"])
        save(l, "31_load_lag_by_rail")

        spread = l.p50.max() - l.p50.min() if l.p50.notna().any() else None
        record("load_lag_p50_spread_days", None if spread is None else float(spread))
        print(f"\nspread of median lag across rails: {spread} days")
        print("  ≥2 days  → rail-dependent → trans_dt behaves like an EVENT date")
        print("  ~0 days  → flat → trans_dt behaves like a POSTING date")
        print("  any material negative lag → forward-dated EFFECTIVE date")

In [ ]:
with step("3.2 day-of-week profile by rail"):
    dow = (TX.withColumn("txn_d", F.to_date(F.col("trans_dt").cast("string")))
             .withColumn("dow", F.date_format("txn_d", "E"))
             .groupBy("payment_rail", "dow")
             .agg(F.count(F.lit(1)).alias("rows"),
                  F.sum(F.col("trans_amt").cast("double")).alias("dollars")))
    d = to_pd(dow)
    piv = d.pivot_table(index="payment_rail", columns="dow", values="rows",
                        aggfunc="sum", fill_value=0)
    order = [x for x in ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"] if x in piv.columns]
    piv = piv[order]
    share = piv.div(piv.sum(axis=1), axis=0)
    share["weekend_share"] = share[[c for c in ["Sat", "Sun"] if c in share.columns]].sum(axis=1)
    share = share.sort_values("weekend_share", ascending=False).reset_index()
    save(share, "32_dow_by_rail")
    record("weekend_share_by_rail",
           {r.payment_rail: round(float(r.weekend_share), 4) for r in share.itertuples()})
    print("\n  A rail at ~0% weekend share is posting on banking days. A rail at")
    print("  ~28% (2/7) is recording the event. Mixed values across rails mean the")
    print("  column's semantics are NOT uniform and Q2 has no single answer.")

In [ ]:
with step("3.3 day-of-month profile"):
    dom = (TX.withColumn("dom", F.dayofmonth(F.to_date(F.col("trans_dt").cast("string"))))
             .groupBy("dom")
             .agg(F.count(F.lit(1)).alias("rows"),
                  F.sum(F.col("trans_amt").cast("double")).alias("dollars"))
             .orderBy("dom"))
    dm = to_pd(dom)
    dm["share_rows"] = dm.rows / dm.rows.sum()
    save(dm, "33_day_of_month")
    print("\n  1st/15th/last-day spikes are payroll and billing cycles — real behaviour.")
    print("  A single dominant day is a batch artefact and would mean daily grain is")
    print("  not available regardless of what trans_dt means.")

> **What this phase cannot settle.** None of these tests distinguishes
> *initiation* from *effective* date; both are event-like. If the load lag is
> rail-dependent and weekend mass tracks the rail's operating calendar, then
> `trans_dt` is an event date and as-of feature construction is possible — which
> is the decision Q2 is actually gating. The initiation-vs-effective distinction
> only matters for settlement-lag analysis, which needs the second date column
> the table does not have, and that request should go upstream with the evidence
> from §3.1 attached.

---

# Phase 4 · The unprofiled columns

**Open questions 4, 5, 6, 7, 9.** A generic profiler first, then the specific
tests where a hypothesis exists worth falsifying.

In [ ]:
def profile(df, c, n_total, topn=TOP_N, dollars=True):
    """cardinality, null/blank share, top values by rows and dollars."""
    if c not in df.columns:
        print(f"  ({c} absent)")
        return None, None
    agg = df.agg(
        F.sum(F.col(c).isNull().cast("long")).alias("nulls"),
        F.sum((~present(c)).cast("long")).alias("absent"),
        F.approx_count_distinct(c).alias("approx_distinct")).first()
    summary = pd.DataFrame([{
        "column": c, "rows": n_total,
        "null_share": agg["nulls"] / n_total,
        "absent_share": agg["absent"] / n_total,
        "approx_distinct": agg["approx_distinct"],
        "rows_per_value": n_total / max(agg["approx_distinct"], 1),
        "reading": ("IDENTIFIER" if agg["approx_distinct"] > 0.5 * n_total else
                    "HIGH-CARD" if agg["approx_distinct"] > 10000 else
                    "CLASSIFIER")}])
    a = [F.count(F.lit(1)).alias("rows")]
    if dollars and "trans_amt" in df.columns:
        a.append(F.sum(F.col("trans_amt").cast("double")).alias("dollars"))
    top = to_pd(df.groupBy(c).agg(*a).orderBy(F.desc("rows")).limit(topn))
    top["share_rows"] = top.rows / n_total
    return summary, top


UNPROFILED = cols("np_key", "trans_purpose", "zelle_recipient_token", "card_entry_mode",
                  "unq_cpty_id", "mdm_id_pays_orig", "mdm_id_receives_orig",
                  "originating_company", "merchant_cat_cd", "cpty_type", "src_syst",
                  "payment_rail", "category_prefix", "trans_currency")

with step("4.1 column profiles"):
    summaries, tops = [], {}
    for c in UNPROFILED:
        s, t = profile(TX, c, N_ROWS)
        if s is not None:
            summaries.append(s)
            tops[c] = t
    prof = pd.concat(summaries, ignore_index=True)
    save(prof, "40_column_profiles")
    record("column_profiles", prof.set_index("column")["reading"].to_dict())

In [ ]:
with step("4.1b top values"):
    for c, t in tops.items():
        if t is None or len(t) == 0:
            continue
        print(f"\n── {c} " + "─" * max(0, 40 - len(c)))
        display(t.head(12))
        t.to_csv(os.path.join(QA, f"40_top_{c}.csv"), index=False)

## 4.2 · Q4 + Q9 — `_orig` and the third-party-originator reading

Two questions, one test. The hypothesis: `mdm_id_pays_orig` differs from
`mdm_id_pays` exactly when a third party originated the payment, and `category`
marks those rows `wTPO`. If the disagreement rate is ~100% on `wTPO` and ~0% on
`woTPO`, both questions are answered at once and `_orig` becomes usable as the
true originator for attribution.

If `_orig` disagrees at similar rates on both, it means something else — most
likely an MDM survivorship artefact (pre- vs post-merge id), which is worth
knowing but is not a payment concept.

In [ ]:
with step("4.2 _orig vs base id, against wTPO"):
    if not has("mdm_id_pays_orig", "category"):
        print("columns absent — skipping")
    else:
        t = TX.withColumn(
            "tpo", F.when(F.col("category").rlike("(?i)woTPO"), F.lit("woTPO"))
                    .when(F.col("category").rlike("(?i)wTPO"), F.lit("wTPO"))
                    .otherwise(F.lit("n/a")))
        for side in ["pays", "receives"]:
            b, o = f"mdm_id_{side}", f"mdm_id_{side}_orig"
            if not has(b, o):
                continue
            t2 = t.withColumn(
                f"cmp_{side}",
                F.when(~present(b) & ~present(o), "both_absent")
                 .when(present(b) & ~present(o), "orig_absent")
                 .when(~present(b) & present(o), "base_absent")
                 .when(F.col(b) == F.col(o), "equal")
                 .otherwise("DIFFER"))
            x = to_pd(t2.groupBy("tpo", f"cmp_{side}")
                        .agg(F.count(F.lit(1)).alias("rows"))
                        .orderBy("tpo", F.desc("rows")))
            x["share_within_tpo"] = x.rows / x.groupby("tpo").rows.transform("sum")
            save(x, f"42_orig_vs_base_{side}")

        # does originating_company populate on wTPO only?
        if has("originating_company"):
            oc = to_pd(t.groupBy("tpo")
                        .agg(F.count(F.lit(1)).alias("rows"),
                             F.avg(present("originating_company").cast("int"))
                              .alias("originating_company_populated")))
            save(oc, "42_originating_company_by_tpo")
            print("\n  originating_company populated ~only on wTPO confirms the")
            print("  third-party-originator reading independently of the _orig columns.")

## 4.3 · Q5 — `unq_cpty_id` vs `unq_cpty_acct_id`

The seed's hypothesis: `unq_cpty_id` may be an entity-level roll-up of the
account-level key, in which case it addresses part of the 91%-singleton fan-in
problem in §4. Two things have to hold for that to be true, and they are
different: the mapping must be many-accounts-to-one-entity, **and** the roll-up
must actually raise the share of counterparties seen by more than one PNC
customer. A roll-up that merges accounts belonging to the same customer changes
nothing about fan-in.

Both are measured. The second is the one that matters.

In [ ]:
with step("4.3 counterparty key hierarchy"):
    if not has("unq_cpty_id", "unq_cpty_acct_id"):
        print("columns absent — skipping")
    else:
        both = TX.filter(present("unq_cpty_id") & present("unq_cpty_acct_id"))
        pairs = both.select("unq_cpty_id", "unq_cpty_acct_id").distinct()   # stage 1
        a = to_pd(pairs.groupBy("unq_cpty_id")
                       .agg(F.count(F.lit(1)).alias("n_acct"))
                       .groupBy("n_acct").agg(F.count(F.lit(1)).alias("n_entity"))
                       .orderBy("n_acct").limit(30))
        b = to_pd(pairs.groupBy("unq_cpty_acct_id")
                       .agg(F.count(F.lit(1)).alias("n_entity"))
                       .groupBy("n_entity").agg(F.count(F.lit(1)).alias("n_acct"))
                       .orderBy("n_entity").limit(30))
        save(a, "43_accounts_per_entity")
        save(b, "43_entities_per_account")
        n_e = both.select(F.approx_count_distinct("unq_cpty_id").alias("d")).first()["d"]
        n_a = both.select(F.approx_count_distinct("unq_cpty_acct_id").alias("d")).first()["d"]
        print(f"\ndistinct unq_cpty_id      : {n_e:,}")
        print(f"distinct unq_cpty_acct_id : {n_a:,}")
        print(f"compression               : {n_a / max(n_e, 1):.3f} accounts per entity")
        record("cpty_entity_compression", round(n_a / max(n_e, 1), 4))

In [ ]:
with step("4.3b fan-in under both keys — the payoff"):
    if not has("unq_cpty_id", "unq_cpty_acct_id"):
        print("columns absent — skipping")
    else:
        ego = F.coalesce(F.col("mdm_id_pays"), F.col("mdm_id_receives"))
        rows = []
        for key in ["unq_cpty_acct_id", "unq_cpty_id"]:
            # two-stage reduce: distinct (cpty, ego) first, then count — §7
            p = (TX.filter(present(key)).select(F.col(key).alias("k"), ego.alias("ego"))
                   .filter(F.col("ego").isNotNull()).distinct())
            fan = p.groupBy("k").agg(F.count(F.lit(1)).alias("n_egos"))
            agg = fan.agg(F.count(F.lit(1)).alias("n_cpty"),
                          F.avg((F.col("n_egos") == 1).cast("double")).alias("share_singleton"),
                          F.avg((F.col("n_egos") >= 5).cast("double")).alias("share_ge5"),
                          F.avg("n_egos").alias("mean_egos")).first()
            rows.append({"key": key, **{k: agg[k] for k in
                         ["n_cpty", "share_singleton", "share_ge5", "mean_egos"]}})
        fi = pd.DataFrame(rows)
        save(fi, "43_fanin_by_key")
        record("fanin", fi.set_index("key")[["share_singleton", "share_ge5"]]
                          .round(4).to_dict("index"))
        print("\n  Seed baseline (acct key, 3 months): 91% singleton, 0.33% seen by ≥5.")
        print("  If the entity key does not move share_ge5 materially, the roll-up")
        print("  does NOT solve the fan-in problem and Q5 closes negative.")

## 4.4 · Q6 — `np_key`

Answered by the `rows_per_value` column in §4.1 and by whether it appeared as a
unique key in §1.6. Two further checks: is it stable for a given counterparty
(a counterparty attribute) or for a given transaction (an identifier)?

In [ ]:
with step("4.4 np_key"):
    if not has("np_key"):
        print("absent — skipping")
    else:
        nk = TX.filter(present("np_key"))
        n_nk = nk.count()
        if n_nk == 0:
            print("np_key is never populated in this slice — Q6 closes: unused column")
        else:
            r = nk.agg(
                F.approx_count_distinct("np_key").alias("d_np"),
                F.approx_count_distinct("trans_id").alias("d_txn")).first()
            print(f"populated rows      : {n_nk:,} ({n_nk / N_ROWS:.1%})")
            print(f"distinct np_key     : {r['d_np']:,}")
            print(f"distinct trans_id   : {r['d_txn']:,}")
            print(f"np_key per txn      : {r['d_np'] / max(r['d_txn'], 1):.3f}")
            # is np_key constant within a counterparty?
            if has("unq_cpty_acct_id"):
                v = (nk.filter(present("unq_cpty_acct_id"))
                       .groupBy("unq_cpty_acct_id")
                       .agg(F.approx_count_distinct("np_key").alias("n"))
                       .agg(F.avg((F.col("n") == 1).cast("double")).alias("share_constant"))
                       .first()["share_constant"])
                print(f"constant within counterparty: {v:.1%}")
                print("  high → counterparty attribute; low → per-transaction identifier")
                record("np_key_constant_within_cpty", round(float(v), 4))

---

# Phase 5 · Q8 — does the merchant namespace ever populate `unq_cpty_acct_id`?

The decision this gates: card flow either joins the same graph or sits beside
it. Either is workable; **not knowing** produces a graph where some card spend
is an edge and some is not, which is the one outcome to avoid.

Three tests, in increasing strength:

1. Co-occurrence — do the two namespaces ever populate on the same row?
2. Rail conditioning — is `merchant_*` exclusively the card rails?
3. **Value overlap** — do any `merchant_id` strings appear as `unq_cpty_acct_id`
   values? If they do, the namespaces collide and a card merchant and a bank
   account can key identically. That is a correctness bug in any graph built
   over the union, not a modelling choice.

In [ ]:
with step("5.1 namespace co-occurrence"):
    if not has("merchant_id"):
        print("merchant_id absent — skipping Phase 5")
    else:
        x = (TX.withColumn("has_cpty", present("unq_cpty_acct_id").cast("int"))
               .withColumn("has_merch", present("merchant_id").cast("int"))
               .groupBy("payment_rail", "has_cpty", "has_merch")
               .agg(F.count(F.lit(1)).alias("rows"),
                    F.sum(F.col("trans_amt").cast("double")).alias("dollars"))
               .orderBy("payment_rail", F.desc("rows")))
        m = to_pd(x)
        m["share"] = m.rows / m.rows.sum()
        save(m, "51_merchant_cpty_cooccurrence")
        both = m[(m.has_cpty == 1) & (m.has_merch == 1)].rows.sum()
        print(f"\nrows carrying BOTH keys: {both:,} ({both / N_ROWS:.3%})")
        record("rows_with_both_namespaces", int(both))

In [ ]:
with step("5.2 value overlap between namespaces"):
    if not has("merchant_id", "unq_cpty_acct_id"):
        print("skipping")
    else:
        mids = TX.filter(present("merchant_id")).select(
            F.col("merchant_id").cast("string").alias("k")).distinct()
        cids = TX.filter(present("unq_cpty_acct_id")).select(
            F.col("unq_cpty_acct_id").cast("string").alias("k")).distinct()
        n_m, n_c = mids.count(), cids.count()
        n_x = mids.join(cids, "k", "inner").count()
        print(f"distinct merchant_id       : {n_m:,}")
        print(f"distinct unq_cpty_acct_id  : {n_c:,}")
        print(f"values in BOTH namespaces  : {n_x:,}")
        record("namespace_value_overlap", n_x)
        if n_x > 0:
            print("\n*** NAMESPACE COLLISION ***")
            print("    Card merchants and bank accounts can key identically. Any graph")
            print("    over the union of both needs a prefixed key (MERCH:<id> /")
            print("    CPTY:<id>), on the same principle as PNC:<mdm_id> in §0.")
            display(to_pd(mids.join(cids, "k", "inner").limit(10)))
        else:
            print("\n    Namespaces are disjoint. A union key is safe, though still")
            print("    worth prefixing for readability.")

In [ ]:
with step("5.3 merchant field completeness"):
    mc = cols("merchant_id", "merchant_city", "merchant_state",
              "merchant_zip_cd", "merchant_cat_cd")
    if not mc:
        print("skipping")
    else:
        card = TX.filter(F.col("payment_rail").rlike("(?i)CARD|PCARD"))
        n_card = card.count()
        if n_card:
            comp = to_pd(card.agg(*[F.avg(present(c).cast("double")).alias(c) for c in mc]))
            comp = comp.T.reset_index()
            comp.columns = ["field", "populated_share_on_card_rails"]
            save(comp, "53_merchant_completeness")
            print(f"\n  merchant_state / _zip_cd give card spend a GEOGRAPHY that the")
            print(f"  off-us counterparty namespace does not have. That is directly")
            print(f"  usable as ground truth for the locatability work — a located")
            print(f"  counterparty with a known position, drawn from the right population.")

---

# Phase 6 · Q10 — non-USD policy

~0.03% of rows, but CAD/EUR/CHF carry real dollars. The policy question
(convert / exclude / report separately) has a precondition the seed does not
state: **is `trans_amt` denominated in `trans_currency`, or already converted to
USD?** The schema discovery in §0.3 looked for a USD-equivalent column. If there
is none, then summing `trans_amt` across currencies is adding CAD to USD, and
the $1.05B / $1.31B figures in the seed are in native units, not dollars.

Absent an FX column there is only one defensible policy: **report separately,
never pool**. Conversion needs a rate table with an as-of date, and §3 has not
even established what the date means yet.

In [ ]:
with step("6.1 currency profile"):
    cur = (TX.groupBy("trans_currency")
             .agg(F.count(F.lit(1)).alias("rows"),
                  F.sum(F.col("trans_amt").cast("double")).alias("amt_native_units"),
                  F.percentile_approx(F.col("trans_amt").cast("double"), 0.5).alias("p50"),
                  F.percentile_approx(F.col("trans_amt").cast("double"), 0.99).alias("p99"),
                  F.approx_count_distinct("trans_id").alias("txns"))
             .orderBy(F.desc("rows")))
    c = to_pd(cur)
    c["share_rows"] = c.rows / c.rows.sum()
    save(c, "60_currency")
    record("currency_share",
           {str(r.trans_currency): round(float(r.share_rows), 6) for r in c.itertuples()})

In [ ]:
with step("6.2 what the non-USD flow actually is"):
    nz = TX.filter(present("trans_currency") & (F.upper("trans_currency") != "USD"))
    if nz.limit(1).count() == 0:
        print("no non-USD rows in this slice")
    else:
        for dim in cols("payment_rail", "cpty_type", "cpty_fin_entity_name", "topo"):
            t = to_pd(nz.groupBy("trans_currency", dim)
                        .agg(F.count(F.lit(1)).alias("rows"),
                             F.sum(F.col("trans_amt").cast("double")).alias("amt"))
                        .orderBy(F.desc("rows")).limit(20))
            print(f"\n── non-USD by {dim} " + "─" * 30)
            display(t)
            t.to_csv(os.path.join(QA, f"62_nonusd_by_{dim}.csv"), index=False)
        print("\n  If non-USD is ~entirely WIRE to foreign fin entities, it is a")
        print("  distinct product with its own analytics and 'report separately' is")
        print("  not a compromise — it is the correct model.")

---

# Phase 7 · Taxonomy assertions

Assert the taxonomy rather than rediscover it (§7). Once the value sets are
known, an unexpected value must fail loudly — a silently-added rail becomes a
silently-dropped slice everywhere downstream.

In [ ]:
EXPECTED_VALUES = {
    "payment_rail": {"ACH", "CHECK", "RTP_PRT", "PCARD", "WIRE",
                     "DEBIT_CARD_SIGNATURE", "RTP_P2P"},
    "cpty_type":    {"CPTY", "NON_PNC_ACH_ORIGINATOR", "MERCHANT", "P2P_CPTY"},
    "src_syst":     {"PME", "PRT-CHIPS", "PRT", "DPI", "RTP"},   # plus NULL
    "topo":         {"INTERNAL_C2C", "INBOUND", "OUTBOUND", "ORPHAN"},
}

with step("7.1 value-set assertions"):
    problems = []
    for c, exp in EXPECTED_VALUES.items():
        if c not in TX.columns:
            continue
        seen = {r[0] for r in TX.select(c).distinct().collect() if r[0] is not None}
        new = seen - exp
        gone = exp - seen
        print(f"{c:16s} seen={len(seen):3d}  unexpected={sorted(new) or '-'}  "
              f"absent={sorted(gone) or '-'}")
        if new:
            problems.append(f"{c}: unexpected values {sorted(new)}")
    record("taxonomy_problems", problems)

In [ ]:
with step("7.2 invariants"):
    inv = to_pd(TX.agg(
        F.sum((F.col("topo") == "ORPHAN").cast("long")).alias("orphan_rows"),
        F.sum((F.col("trans_amt") <= 0).cast("long")).alias("nonpositive_amt"),
        F.sum(F.col("trans_amt").isNull().cast("long")).alias("null_amt"),
        F.sum((~F.col("trans_dt").cast("string").rlike(r"^\d{4}-\d{2}-\d{2}"))
              .cast("long")).alias("bad_trans_dt"),
        F.sum((F.col("mdm_id_pays") == F.col("mdm_id_receives")).cast("long")).alias("self_loops"),
    ))
    save(inv, "70_invariants")
    for k, v in inv.iloc[0].items():
        if v and k != "self_loops":
            problems.append(f"{k} = {int(v):,} (expected 0)")
    print("\n  self_loops are EXPECTED — one customer moving between its own PNC")
    print("  accounts (§3.1). Not payments; exclude from flow metrics, retain as the")
    print("  only on-us analogue of off-us self-payment.")

    if problems:
        msg = "TAXONOMY / INVARIANT FAILURES:\n  " + "\n  ".join(problems)
        if FAIL_ON_TAXONOMY:
            raise AssertionError(msg)
        print("\n!! " + msg)
    else:
        print("\n  all assertions pass")

---

# Phase 8 · Summary

Everything headline in one file, keyed to the config that produced it. This is
the artefact to paste into the seed document as v2, alongside the per-question
verdicts.

In [ ]:
with step("8 summary"):
    RESULTS["_config"] = {
        "TXN_TABLE": TXN_TABLE, "CUST_DIM": CUST_DIM,
        "PROFILE_MONTHS": PROFILE_MONTHS, "RANGE_MONTHS": RANGE_MONTHS,
        "DIAG_TXN_SAMPLE_PCT": DIAG_TXN_SAMPLE_PCT, "POPULATION": POPULATION,
        "run_at": time.strftime("%Y-%m-%d %H:%M:%S"),
    }
    path = os.path.join(WORK, "summary.json")
    with open(path, "w") as f:
        json.dump(RESULTS, f, indent=2, default=str)
    print("→", path)
    print(json.dumps({k: v for k, v in RESULTS.items()
                      if not isinstance(v, (dict, list))}, indent=2, default=str))

## Verdict template

Fill from the outputs above and carry into `PKG_TXN_TABLE_EDA_SEED.md` v2.

| Q | Question | Verdict | Evidence |
|---|---|---|---|
| 1 | `trans_id` duplication | | §1.0 `dim_rows_per_mdm_id`, §1.1 ratio, §1.2b splitting axis, §1.4b value pairs, §1.5 mixed-topology share |
| 2 | `trans_dt` semantics | | §3.1 lag spread, §3.2 weekend share by rail |
| 3 | History depth | | §2.1 range, §2.2 partial months and `rows_per_txn` steps |
| 4 | `_orig` columns | | §4.2 disagreement × `wTPO` |
| 5 | `unq_cpty_id` roll-up | | §4.3 compression, §4.3b `share_ge5` movement |
| 6 | `np_key` | | §4.1 `rows_per_value`, §4.4 constancy, §1.6 |
| 7 | `trans_purpose` etc. | | §4.1 profiles |
| 8 | Merchant namespace | | §5.1 co-occurrence, §5.2 value overlap |
| 9 | `wTPO` / `woTPO` | | §4.2 `originating_company` by TPO |
| 10 | Non-USD policy | | §0.3 FX columns, §6.1, §6.2 |

## What to run second

Two things this notebook deliberately does not do.

**Do not rebuild the leg table until §1.9 resolves.** If §1.5 fires, the leg
model is wrong rather than mis-fed, and rebuilding against the current model
would bake the error in. The rename (`leg` → `ego_row`) comes before any new
code, per §0 of the seed.

**Phase 5's merchant geography is the surprise worth chasing.** If
`merchant_state` and `merchant_zip_cd` populate on card rails, that is a set of
counterparties with a **known position**, drawn from the right population — the
external ground truth `PKG_LOCATABILITY_MANIFEST` §8 records as unavailable. It
is skewed toward retail merchants and says nothing about a manufacturer's
counterparty, but it is a far larger and less financial-institution-skewed
validation set than the FDIC/NCUA registry, and it costs nothing to check.